In [ ]:
import numpy as np
import pickle
import xgboost as xgb
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import norm
from data_generation import m_0, g_0, get_data
from dml_algorithm import mm_ate, dml_ate

In [ ]:
rng = np.random.default_rng(seed=78)

In [ ]:
N = 16000
y_data, d_data, x_data = get_data(N, rng)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
x_quad_data = poly_features.fit_transform(x_data)
y_train, y_test, d_train, d_test, x_train, x_test, x_quad_train, x_quad_test = train_test_split(y_data, d_data, x_data, x_quad_data, test_size=0.2, random_state=42)

In [ ]:
model_g0 = xgb.XGBRFRegressor(max_depth=6, subsample=0.8, colsample_bynode=0.7, n_estimators=1000, reg_lambda=1, reg_alpha=0.01, objective='reg:squarederror', seed=0)
model_g1 = xgb.XGBRFRegressor(max_depth=6, subsample=0.8, colsample_bynode=0.7, n_estimators=1000, reg_lambda=1, reg_alpha=0.01, objective='reg:squarederror', seed=0)
model_m = xgb.XGBRFClassifier(max_depth=6, subsample=0.8, colsample_bynode=0.7, n_estimators=1000, reg_lambda=1, reg_alpha=0.01, objective='binary:logistic', seed=0)
model_m_cal = CalibratedClassifierCV(estimator=model_m)

In [ ]:
%%time
model_m.fit(x_train, d_train)
print(model_m.predict_proba(x_test)[:20,1])
#model_m_cal.fit(x_train, d_train)
#print(model_m_cal.predict_proba(x_test)[:20,1])
#model_m.fit(x_quad_train, d_train)
#print(model_m.predict_proba(x_quad_test)[:20,1])
print(m_0(x_test[:20]))

In [ ]:
%%time
model_g0.fit(x_train[d_train==0], y_train[d_train==0])
print(model_g0.predict(x_test[:20]))
#model_g0.fit(x_quad_train[d_train==0], y_train[d_train==0])
#print(model_g0.predict(x_quad_test[:20]))
print(g_0(0, x_test[:20]))

In [ ]:
%%time
model_g1.fit(x_train[d_train==1], y_train[d_train==1])
print(model_g1.predict(x_test[:20]))
#model_g1.fit(x_quad_train[d_train==1], y_train[d_train==1])
#print(model_g1.predict(x_quad_test[:20]))
print(g_0(1, x_test[:20]))

In [ ]:
%%time
model_g = [model_g0, model_g1]
dml_ate(y_data, d_data, x_data, model_g, model_m)